# 《PythAPCS123》單元 13-2：語法錯誤（SyntaxError）深度排查與 Traceback 閱讀心法

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-2_syntax_errors_and_traceback_decoding.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：克服初學者面對直譯器紅色英文報錯（Traceback）的內心恐懼，徹底搞懂「錯誤行號（Line Number）」與「游標指標（^）」的真實座標意義，地毯式拆解考場最常見的漏冒號、括號引號不對稱、上一行漏關括號引爆下一行報錯、縮排空格混用、賦值等號誤用及全形標點符號等五大高頻語法地雷，練就 10 秒精準鎖定並修復 SyntaxError 的必備除錯神技。


### 13.2.1 直譯器編譯檢查期：為什麼程式碼尚未執行就先報錯？

許多初學者在剛開始學習 Python 時，常常存在一種根深蒂固的直覺誤解：「Python 是直譯語言，所以它一定是一行一行讀取、讀到哪一行就執行哪一行；因此，只要我的程式前面幾行沒寫錯，電腦至少應該先把前幾行的 `print()` 印出來，等跑到出錯的那一行才會當機停下來吧？」

當初學者滿懷期待地按下執行鍵，卻發現畫面上「連半個字都沒印出來」，直譯器就立刻在第 10 行噴出紅色的 `SyntaxError` 時，往往會感到無比困惑。這是因為 Python 在執行任何程式碼之前，內部必須先經歷一段極其迅速的**「語法剖析與位元組編譯期（Parsing & Bytecode Compilation Phase）」**。直譯器會先把整份原始程式碼（.py 檔案）當作一篇完整的文章從頭到尾掃描一遍，並將人類看得懂的程式文字轉換為電腦底層能夠高效處理的「位元組碼（Bytecode）」。

只要整篇文章中有任何一個單字拼錯、括號沒關、縮排錯位或漏掉冒號，語法剖析器（Parser）就會認定整份文法結構不合規格，直譯器會當機立斷「拒絕啟動虛擬機（Python Virtual Machine, PVM）」，連第 1 行的 `print("程式開始")` 都不會執行！理解這個靜態檢查階段至關重要：語法錯誤是直譯器在門口就把你攔截下來，它與程式執行到一半崩潰的「執行時期錯誤（Runtime Error）」有著本質上的不同。


In [ ]:
# 13.2.1 程式碼演示：對比「語法錯誤編譯期攔截」vs「執行時期崩潰」
import sys

# 案例 A: 語法錯誤（SyntaxError）—— 整份程式連第 1 行都不會執行！
syntax_error_code = '''
print("[步驟 1] 程式準備啟動")
print("[步驟 2] 讀取環境變數")
# 第 5 行故意漏掉冒號
if 10 > 5
    print("[步驟 3] 條件成立")
'''

print("=== 案例 A: 執行包含 SyntaxError 的代碼 ===")
try:
    exec(syntax_error_code)
except SyntaxError as e:
    print(f"❌ 攔截到 SyntaxError！錯誤發生於第 {e.lineno} 行: {e.msg}")
    print("觀察重點：注意上方完全沒有印出『步驟 1』或『步驟 2』！編譯期即被攔截。")

# 案例 B: 執行時期錯誤（RuntimeError）—— 前面正常印出，直到出錯行才崩潰！
runtime_error_code = '''
print("[步驟 1] 程式順利啟動")
print("[步驟 2] 正在進行前置計算")
# 語法完全正確，但數值邏輯引爆除以零崩潰
danger_val = 10 / 0
print("[步驟 3] 運算完成")
'''

print("\n=== 案例 B: 執行包含 RuntimeError 的代碼 ===")
try:
    exec(runtime_error_code)
except ZeroDivisionError as e:
    print(f"⚠️ 攔截到 ZeroDivisionError: {e}")
    print("觀察重點：步驟 1 與步驟 2 均已成功印出，直到第 6 行才在執行階段中途暴斃！")

# 案例 C: 多重語法錯誤掩蓋現象 —— 直譯器永遠只會回報「第一個撞到的錯誤」！
multi_error_code = '''
if x > 0       # 錯誤 1: 漏冒號
    print(x)
for i in range(5) # 錯誤 2: 漏冒號
    print(i)
'''
print("\n=== 案例 C: 多重語法錯誤只報第一個 ===")
try:
    compile(multi_error_code, "<multi_test>", "exec")
except SyntaxError as e:
    print(f"直譯器回報行號: 第 {e.lineno} 行（錯誤 1）")
    print("提示：修復了第 2 行的錯誤後，再次編譯才會暴露第 4 行的錯誤 2！")


### 13.2.1 語法重點回顧與核心觀念提煉

在剛才的三重案例驗證中，我們建立了兩大極具價值的除錯心智模型：
1. **編譯期攔截 vs 執行期中斷**：
   - `SyntaxError`（語法錯誤）：發生於編譯期，整份程式連第 1 行都進不去，在 OJ 評判系統直接拿下 `CE（Compile Error）`。
   - `Exception`（執行時期例外）：語法完全合法，程式能正常啟動並執行前段，直到出錯那一行才中途崩潰，OJ 判定為 `RE（Runtime Error）`。
2. **連鎖排查定律**：因為直譯器在遇到第一個語法錯誤時就會立刻停止掃描，所以當你修復了報錯的第 5 行後，再次執行可能又在第 15 行跳出新報錯。這是完全正常的現象，切勿慌張，只需順著 Traceback 由上而下逐一消滅即可！


In [ ]:
# 13.2.1 學生實作練習：程式碼執行階段診斷器
# 任務說明：實作 diagnose_code_phase(code_str) 函式
# 判斷傳入的 code_str 屬於哪種階段狀態：
# 1. 若語法檢查失敗（引發 SyntaxError），回傳 "COMPILE_ERROR"
# 2. 若能順利通過 compile() 編譯，回傳 "SYNTAX_OK"

def diagnose_code_phase(code_str: str) -> str:
    # 請在此處使用 compile 搭配 try-except 進行階段判定
    try:
        compile(code_str, filename="<test>", mode="exec")
        return "SYNTAX_OK"
    except SyntaxError:
        return "COMPILE_ERROR"

# 測試用例
code_valid = "a = 10\nb = 20\nprint(a + b)"
code_invalid = "for i in range(10)\n    print(i)"
print("測試 1 (合法代碼):", diagnose_code_phase(code_valid))
print("測試 2 (漏冒號代碼):", diagnose_code_phase(code_invalid))


In [ ]:
# 13.2.1 單元測試驗證
assert diagnose_code_phase("x = 5\nprint(x)") == "SYNTAX_OK"
assert diagnose_code_phase("if True\n    pass") == "COMPILE_ERROR"
assert diagnose_code_phase("arr = [1, 2, 3") == "COMPILE_ERROR"
assert diagnose_code_phase("def foo(x):\n    return x * 2") == "SYNTAX_OK"
assert diagnose_code_phase("print('hello' + 5)") == "SYNTAX_OK", "語法合法，型態錯誤屬於執行期而非語法期"
print("🎉 13.2.1 所有測試通過！成功掌握編譯期與執行期核心機制！")


### 13.2.2 看懂 Traceback 座標軸：錯誤行號（Line number）與游標指標 `^` 定位

每當終端機噴出一整片紅色的英文錯誤追蹤訊息（Traceback）時，許多零基礎學生的第一反應往往是「眼神迴避、心跳加速、不知所措」，甚至下意識立刻把整個視窗關掉重開。請記住：**Traceback 不是電腦在罵你，而是電腦在向你發出最精準的求救訊號！**

在所有語法錯誤的 Traceback 訊息中，直譯器提供了一套非常精準的「二維空間座標定位系統」：
1. **第一維度：檔案名稱與行號（`File "...", line X`）**：這是直譯器的 Y 軸座標。它明確告訴你，解析器是在第幾行撞牆的。
2. **第二維度：原始程式碼片段（Code Text）**：直譯器會把出問題的那一行程式碼完整印出來給你看。
3. **第三維度：游標箭頭指標（Caret Symbol `^` 或波浪線 `~~~`）**：這是直譯器的 X 軸座標。在 Python 3.10+ 版本中，強化版直譯器會用一個或多個 `^` 與 `~`，極其精準地指著它「看到哪一個字元發現不合法」。
4. **第四維度：錯誤類型與原因描述（Error Message）**：在最底下一行，如 `SyntaxError: expected ':'`（預期需要冒號）或 `SyntaxError: unmatched ')'`（不匹配的多餘右括號）。

⚠️ **考場最大盲區：上一行漏括號，下一行背黑鍋！**
如果上一行的小括號、中括號或引號沒有關閉（例如 `total = (10 + 20`），Python 直譯器會認為該表達式尚未結束，繼續往下讀取第二行；當它讀到第二行的第一個指令時，才會發現文法完全接不起來，於是在第二行噴出 `SyntaxError`！這時**游標指著第二行，但真正的罪魁禍首 100% 在上一行**！


In [ ]:
# 13.2.2 程式碼演示：解析 Traceback 座標軸與「上一行未閉合」的嫁禍陷阱
import traceback

def analyze_syntax_traceback(code_text, label):
    print(f"=== 分析案例 [{label}] ===")
    lines = code_text.strip().split('\n')
    for idx, l in enumerate(lines, 1):
        print(f"  行 {idx:02d}: {l}")
    try:
        compile(code_text, "<test_file>", "exec")
        print("  --> [PASS] 語法檢查通過！\n")
    except SyntaxError as e:
        print(f"  --> [直譯器報錯座標] 行號: 第 {e.lineno} 行, 欄位: 第 {e.offset} 字元")
        print(f"  --> [出錯代碼片段] {repr(e.text.strip() if e.text else '')}")
        indent = " " * (e.offset - 1) if e.offset else ""
        print(f"  --> [游標指標箭頭] {indent}^")
        print(f"  --> [診斷訊息描述] {e.msg}")
        if e.lineno > 1 and "invalid syntax" in e.msg:
            print("  💡 [考場避坑提示] 報錯在該行開頭？請務必抬頭檢查『前一行』是否括號或引號未閉合！")
        print()

# 案例 1: 當前行明確出錯（多餘運算子）
analyze_syntax_traceback("a = 10\nb = 20 + * 5\nprint(a + b)", "當前行明確錯誤")

# 案例 2: 經典黑鍋陷阱 —— 第 1 行漏右括號，報錯卻在第 2 行！
analyze_syntax_traceback("total = sum([1, 2, 3\nans = total * 2\nprint(ans)", "上一行未閉合嫁禍陷阱")

# 案例 3: 巢狀呼叫末尾漏括號
analyze_syntax_traceback("val = int(input().split()[0]\nprint(val)", "雙層巢狀呼叫漏括號")


### 13.2.2 語法重點回顧與核心觀念提煉

閱讀 Traceback 的標準五部曲：
1. **看最後一行**：先讀最底部的錯誤描述（如 `SyntaxError: expected ':'`、`SyntaxError: unterminated string literal`），搞懂電腦到底想要什麼。
2. **看行號與箭頭**：找到 `File "...", line X` 與 `^` 箭頭，確認電腦撞牆的精確字元。
3. **啟動「抬頭望上一行」雷達**：如果箭頭指在某個變數的開頭、或是看似毫無問題的指令上，**立刻抬頭看前一行**！90% 的機率是前一行少打了一個右括號 `)`、右中括號 `]` 或引號。
4. **利用現代編輯器的彩虹括號對齊**：在 Colab 或 VSCode 中，將游標點在括號旁邊，觀察對應的配對括號是否有亮起，能在 3 秒內揪出失蹤的括號！


In [ ]:
# 13.2.2 學生實作練習：括號配對平衡檢查器
# 任務說明：實作 is_brackets_balanced(expr) 函式
# 檢查字串 expr 中的小括號 ()、中括號 []、大括號 {} 是否完全對稱且正確閉合
# 若完全合法回傳 True；若有未閉合或類型錯配回傳 False

def is_brackets_balanced(expr: str) -> bool:
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    # 請在此處使用堆疊實作括號平衡檢驗
    for ch in expr:
        if ch in "([{":
            stack.append(ch)
        elif ch in ")]}":
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
    return len(stack) == 0

# 測試用例
print("測試合法:", is_brackets_balanced("[(1 + 2) * 3]"))
print("測試未閉合:", is_brackets_balanced("total = sum([1, 2, 3"))
print("測試錯配:", is_brackets_balanced("[1, 2, 3)"))


In [ ]:
# 13.2.2 單元測試驗證
assert is_brackets_balanced("(1 + 2)") == True
assert is_brackets_balanced("[(a + b) * {c - d}]") == True
assert is_brackets_balanced("((())") == False
assert is_brackets_balanced("[1, 2, 3)") == False
assert is_brackets_balanced(")") == False
assert is_brackets_balanced("") == True
print("🎉 13.2.2 所有測試通過！成功破解括號不對稱與座標定位盲點！")


### 13.2.3 考場高頻語法雷區一：漏冒號 `:` 與括號/引號不對稱

在 APCS 考場的語法錯誤統計中，**「漏打冒號 `:`」**長年位居扣分排行榜的第一名！
在 Python 的文法設計中，冒號具有神聖的地位：它是引導直譯器進入下一個縮排區塊（Suite）的唯一通行證。

考場最容易漏冒號的六大關鍵字組合：
1. **條件分支族**：`if condition:`、`elif condition:`、`else:`（初學者極常漏打 else 後面的冒號）。
2. **迴圈控制族**：`for item in sequence:`、`while condition:`。
3. **函式定義族**：`def function_name(params):`。
4. **例外防禦族**：`try:`、`except Exception:`、`finally:`。

此外，從 C/C++ 或 Java 轉向 Python 的考生，常下意識寫出 `if (x > 0) {` 或 `while (x > 0)` 而忘記在尾端補上冒號。
引號方面，最常發生的問題是**「單雙引號混用未閉合」**（如 `'hello"`）或字串中含有縮寫逗號（如 `'It's cool'` 中的 `'It'` 提早將字串閉合，導致後方的 `s cool'` 變成語法亂碼）。


In [ ]:
# 13.2.3 程式碼演示：漏冒號、C語言習慣與引號混用四大雷區
print("--- 雷區 1: if / elif / else 各環節漏冒號示範 ---")
def demonstrate_colon_pitfall():
    test_cases = [
        ("if x > 10\n    pass", "if 漏冒號"),
        ("elif x == 5\n    pass", "elif 漏冒號"),
        ("else\n    pass", "else 漏冒號 (考場最高頻手滑！)"),
        ("for i in range(5)\n    pass", "for 迴圈漏冒號"),
        ("def calc(a, b)\n    return a + b", "def 函式宣告漏冒號")
    ]
    for code, desc in test_cases:
        try:
            compile(code, "<colon_test>", "exec")
        except SyntaxError as e:
            print(f"  [攔截成功] {desc} -> 報錯行 {e.lineno}: {e.msg}")

demonstrate_colon_pitfall()

print("\n--- 雷區 2: C/C++ 習慣語法在 Python 中引爆語法錯誤 ---")
c_style_code = '''
if (x > 10) {
    print("x is big")
}
'''
try:
    compile(c_style_code, "<c_style>", "exec")
except SyntaxError as e:
    print(f"  [C語言大括號大忌] 報錯: {e.msg} (Python 用縮排而非大括號 {{}} 界定區塊！)")

print("\n--- 雷區 3: 引號閉合衝突與字串內部單引號陷阱 ---")
quote_trap_code = "msg = 'It's a sunny day'" # 內部撇號提前閉合了字串
try:
    compile(quote_trap_code, "<quote_test>", "exec")
except SyntaxError as e:
    print(f"  [引號提前閉合] 報錯: {e.msg}")
    print("  正解：外層使用雙引號，或使用跳脫字元！")


### 13.2.3 語法重點回顧與核心觀念提煉

冒號與引號防身守則：
1. **只要新開區塊，必定以冒號收尾**：打完 `if`、`elif`、`else`、`for`、`while`、`def` 的當下，右手小指立刻按下鍵盤上的 `:` 鍵，養成肌肉記憶。
2. **引號三層防護策**：
   - 含有單引號（撇號）的英文字句，外層一律使用雙引號：`"He said it's OK"`。
   - 跨多行的大段文字，一律使用三引號：`'''...'''`。
   - 絕不把中文全形引號（如 `“` 與 `”`）當作程式碼引號。


In [ ]:
# 13.2.3 學生實作練習：自動補齊語法結構冒號
# 任務說明：實作 auto_fix_colons(line) 函式
# 給定單行程式碼字串 line（例如 "if x > 5" 或 "    else"）
# 若該行屬於需要冒號結尾的複合語句開頭（以 if, elif, else, for, while, def 開頭）
# 且結尾尚未包含冒號 ':'，請自動在末尾補上 ':'；其餘情況保持原樣回傳！

def auto_fix_colons(line: str) -> str:
    stripped = line.rstrip()
    leading_spaces = line[:len(line) - len(line.lstrip())]
    pure_text = stripped.lstrip()
    
    keywords = ("if ", "elif ", "else", "for ", "while ", "def ")
    needs_colon = any(pure_text.startswith(kw) or pure_text == kw.strip() for kw in keywords)
    
    if needs_colon and not pure_text.endswith(":"):
        return stripped + ":"
    return line

# 測試用例
print("修復 if:", auto_fix_colons("if x > 10"))
print("修復 else (帶縮排):", repr(auto_fix_colons("    else")))
print("已有冒號:", auto_fix_colons("for i in range(5):"))


In [ ]:
# 13.2.3 單元測試驗證
assert auto_fix_colons("if x == 1") == "if x == 1:"
assert auto_fix_colons("    else") == "    else:"
assert auto_fix_colons("def foo(x):") == "def foo(x):"
assert auto_fix_colons("x = 10") == "x = 10"
assert auto_fix_colons("while True") == "while True:"
print("🎉 13.2.3 所有測試通過！成功建立冒號與引號肌肉記憶！")


### 13.2.4 考場高頻語法雷區二：縮排不一致 IndentationError 與全形空白災難

在 Python 中，「縮排（Indentation）」不是單純為了排版美觀，而是程式邏輯區塊的唯一標誌。這與 C/C++、Java 使用大括號 `{}` 的機制截然不同。

考場最常引爆縮排災難的三大情境：
1. **`IndentationError: unexpected indent`（意外的多餘縮排）**：在不該縮排的地方多敲了空格，例如在最外層的主程式第 1 行前方手滑按了一個空白鍵。
2. **`IndentationError: expected an indented block`（預期需要縮排區塊）**：在 `if` 或 `def` 冒號下方，留下了空行而沒有寫任何指令。若想暫時留白，必須明確寫下 `pass` 佔位！
3. **全形空白幽靈災難（Full-width Space `\u3000`）**：
   這是所有零基礎初學者最常遭遇的「靈異事件」！當考生從題目的 PDF 說明、教學講義或是 LINE/網頁複製程式碼範例時，文字中經常夾帶了中文全形空白鍵（Unicode 編碼 `\u3000`，ASCII 值為 12288）。
   在螢幕上看，它跟一般英文半形空格長得一模一樣；但 Python 直譯器只認得半形 ASCII 空格（`\x20`，ASCII 值為 32）。一旦遇到全形空白，直譯器會當場引爆令人崩潰的：
   ```
   SyntaxError: invalid character '　' (U+3000)
   ```
   學會識別全形空白並一鍵淨化，是避免考場無謂失分的救命法寶！


In [ ]:
# 13.2.4 程式碼演示：縮排錯誤類型剖析與全形空白隱形幽靈顯形術
print("--- 縮排錯誤案例 1: 開頭意外縮排 unexpected indent ---")
code_indent_1 = " x = 10\nprint(x)" # 第 1 行前方多了一個空格
try:
    compile(code_indent_1, "<indent_test>", "exec")
except IndentationError as e:
    print(f"  [捕捉成功] IndentationError: {e.msg} (行 {e.lineno})")

print("\n--- 縮排錯誤案例 2: 冒號後留空 expected an indented block ---")
code_indent_2 = "if True:\n# 只有註解沒有程式碼\nprint('done')"
try:
    compile(code_indent_2, "<indent_test>", "exec")
except IndentationError as e:
    print(f"  [捕捉成功] IndentationError: {e.msg} (行 {e.lineno})")
    print("  💡 正解：若暫時不寫邏輯，必須在縮排區塊內寫下 pass ！")

print("\n--- 縮排錯誤案例 3: 全形空白幽靈 (U+3000) 顯形術 ---")
# 建立一個包含全形空白的危險字串（常複製自講義或網頁）
fullwidth_space_code = "total　=　100" # 裡面的空格是全形 \u3000

print(f"肉眼觀察字串內容: {repr(fullwidth_space_code)}")
for idx, ch in enumerate(fullwidth_space_code):
    print(f"  字元 [{ch}] -> Unicode 編碼: U+{ord(ch):04X}, ASCII 碼: {ord(ch)}")

try:
    compile(fullwidth_space_code, "<fullwidth_test>", "exec")
except SyntaxError as e:
    print(f"\n  [直譯器報警] SyntaxError: {e.msg}")
    print("  說明：Python 堅決拒絕全形空白！必須使用半形空格 (ASCII 32)。")


### 13.2.4 語法重點回顧與核心觀念提煉

縮排與空白防禦兩大原則：
1. **縮排統一原則**：全面使用「4 個半形空白」或「1 個 Tab」縮排，絕對不要在一份程式碼中忽而按 Tab 忽而按空格，否則在 Python 3 中會直接觸發 `TabError: inconsistent use of tabs and spaces in indentation`。
2. **全形字元淨化心法**：當遇到奇怪的 `invalid character` 報錯時：
   - 將出錯的那一行整行刪除，手動切換為「半形英數輸入法」重新敲一次。
   - 使用快捷鍵全文取代：將全形空白 `　` 取代為半形空白 ` `。


In [ ]:
# 13.2.4 學生實作練習：全形字元掃描與淨化器
# 任務說明：實作 sanitize_fullwidth_chars(source_code) 函式
# 傳入可能包含全形空白或中文標點的程式文字 source_code
# 1. 將全形空白 '\u3000' 替換為半形空格 ' '
# 2. 將全形冒號 '：' 替換為半形冒號 ':'
# 3. 將全形逗號 '，' 替換為半形逗號 ','
# 4. 回傳淨化後的程式碼字串，確保能順利通過 compile！

def sanitize_fullwidth_chars(source_code: str) -> str:
    # 請在此處進行替換淨化
    replacements = {
        '\u3000': ' ',
        '：': ':',
        '，': ',',
        '（': '(',
        '）': ')'
    }
    clean_code = source_code
    for old_ch, new_ch in replacements.items():
        clean_code = clean_code.replace(old_ch, new_ch)
    return clean_code

# 測試用例
dirty_code = "if　x　>　5：\n　　print(x，10)"
print("淨化前:\n" + repr(dirty_code))
print("淨化後:\n" + sanitize_fullwidth_chars(dirty_code))


In [ ]:
# 13.2.4 單元測試驗證
dirty = "a　=　10\nif a > 5：\n    print(a，20)"
cleaned = sanitize_fullwidth_chars(dirty)
assert '\u3000' not in cleaned
assert '：' not in cleaned
assert '，' not in cleaned
# 驗證淨化後的代碼能否順利編譯
compile(cleaned, "<test>", "exec")
print("🎉 13.2.4 所有測試通過！成功掌握縮排規範與全形字元淨化技術！")


### 13.2.5 考場高頻語法雷區三：等號賦值 `=` 與比較 `==` 誤用，及中文標點混淆

在初學者的思維中，數學課本上的「$x = 5$」代表「$x$ 等於 5」。
但在程式語言的世界中：
- **單等號 `=` 是「賦值運算子（Assignment）」**：將右邊的計算結果送入左邊的變數箱子中。
- **雙等號 `==` 才是「等於比較運算子（Equality Comparison）」**：比對兩側的數值是否相等，回傳 `True` 或 `False`。

在 APCS 考場上，初學者在撰寫條件判斷時，極常手滑寫出：
```python
if x = 5: # 致命錯誤！
    print("x 等於 5")
```
在 Python 中，`if` 語句的條件表達式中「嚴格禁止使用單等號賦值」！直譯器會立刻噴出：
```
SyntaxError: invalid syntax. Maybe you meant '==' or ':='?
```
這項錯誤在 Python 3.10 之後有了非常溫馨的提示文字（`Maybe you meant '=='?`），但在考場高壓下，許多學生依舊容易慌亂。

此外，中英文輸入法切換不全也是語法期的大殺手：中文括號 `（）`、中文引號 `“”`、中文驚嘆號 `！=` 與中文逗號 `，`，在直譯器眼中全部都是非法的異星符號！


In [ ]:
# 13.2.5 程式碼演示：單等號誤用診斷與中文標點錯誤對比
print("--- 案例 1: 條件式中誤用單等號賦值 ---")
assign_in_if_code = "if x = 10:\n    pass"

try:
    compile(assign_in_if_code, "<assign_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤] SyntaxError: {e.msg}")
    print(f"  出錯片段: {repr(e.text.strip() if e.text else '')}")
    print("  直譯器貼心提示：電腦在問你是不是手滑把比較 '==' 打成賦值 '=' 了！")

print("\n--- 案例 2: 中文全形括號混入代碼 ---")
chinese_paren_code = "print（'hello'）" # 括號是中文全形 （）
try:
    compile(chinese_paren_code, "<paren_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤] SyntaxError: {e.msg}")
    print("  說明：中文全形括號 (U+FF08) 無法取代半形 ASCII 括號 ()！")

print("\n--- 案例 3: 中文逗號混入引數列表 ---")
chinese_comma_code = "a, b = map(int，input().split())" # 逗號是全形
try:
    compile(chinese_comma_code, "<comma_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤] SyntaxError: {e.msg}")
    print("  說明：中文逗號導致直譯器在切分引數時撞牆崩潰！")


### 13.2.5 語法重點回顧與核心觀念提煉

賦值等號與標點符號防禦心法：
1. **比較雙等號、賦值單等號**：
   - 想要存東西：`score = 100`（單等號）。
   - 想要問問題：`if score == 100:`（雙等號，必加冒號）。
2. **考場輸入法強制設定**：在進考場打開程式編輯器後，**第一件事請將輸入法切換為純英文模式（EN / 半形）**！所有的括號、引號、逗號、冒號一律在英文輸入狀態下敲擊，從源頭杜絕全形標點符號的入侵。


In [ ]:
# 13.2.5 學生實作練習：語法合規賦值與比較驗證器
# 任務說明：實作 validate_if_statement(expr_str) 函式
# 檢查傳入的 if 條件表達式字串（如 "x == 10" 或 "x = 10"）
# 1. 若表達式包含非法的單等號賦值（出現單獨的 '=' 而非 '=='、'!='、'<='、'>='）回傳 "INVALID_ASSIGNMENT"
# 2. 若包含中文全形標點（'：', '，', '（', '）'）回傳 "INVALID_CHINESE_PUNCT"
# 3. 若語法格式合規回傳 "VALID"

def validate_if_statement(expr_str: str) -> str:
    # 檢查中文標點
    for ch in expr_str:
        if ch in "：，（）":
            return "INVALID_CHINESE_PUNCT"
            
    # 檢查是否包含孤立的單等號
    # 巧妙移除合法運算子 ==, !=, <=, >= 後再檢查是否存在 =
    cleaned = expr_str.replace("==", "  ").replace("!=", "  ").replace("<=", "  ").replace(">=", "  ")
    if "=" in cleaned:
        return "INVALID_ASSIGNMENT"
        
    return "VALID"

# 測試用例
print("合法比較:", validate_if_statement("x == 10"))
print("誤用單等號:", validate_if_statement("x = 10"))
print("中文冒號:", validate_if_statement("x == 10："))


In [ ]:
# 13.2.5 單元測試驗證
assert validate_if_statement("score == 100") == "VALID"
assert validate_if_statement("score != 0") == "VALID"
assert validate_if_statement("score <= 60") == "VALID"
assert validate_if_statement("score = 100") == "INVALID_ASSIGNMENT"
assert validate_if_statement("x == 5：") == "INVALID_CHINESE_PUNCT"
assert validate_if_statement("x == （5 + 2）") == "INVALID_CHINESE_PUNCT"
print("🎉 13.2.5 所有測試通過！成功分清賦值與比較，徹底阻斷標點干擾！")


### 13.2.6 語法排查實戰操練：1 分鐘極速修正多重語法硬傷程式碼

在前面的子單元中，我們分別學習了編譯期原理、Traceback 座標軸、冒號括號配對、縮排全形字元以及等號標點誤用。
然而，在真實的 APCS 考場上，一個寫得不太順暢的程式碼片段，往往是**「多種語法錯誤同時交織在一起」**的！

當直譯器一次又一次噴出不同的報錯時，頂尖選手能維持平靜的心跳，遵循一套標準化的排查 SOP：
1. **第一次編譯**：直譯器回報第一個 SyntaxError $\to$ 看行號、看游標、看前一行 $\to$ 立即修正。
2. **第二次編譯**：第一個錯誤消失，直譯器推進到下一個錯誤 $\to$ 重複定位、排查、修正。
3. **第三次編譯**：全數語法錯誤掃除，程式順利進入執行階段！

本節我們將模擬一段包含多處典型高頻語法硬傷的真實學生程式碼，引導你在一分鐘內化身資深代碼醫生，手起刀落逐一修復！


In [ ]:
# 13.2.6 程式碼演示：考場真實多重語法地雷代碼排查修復模擬
# 任務背景：這是一段求 1 到 n 偶數平方和的學生代碼，裡面埋伏了 4 處典型語法硬傷！

buggy_code = '''
def sum_even_squares(n)
    total = 0
    for i in range(1, n + 1:
        if i % 2 == 0:
            total += i ** 2
    return total
'''

print("=== 原始包含語法硬傷的代碼 ===")
print(buggy_code)

def step_by_step_debugger(code):
    current_code = code
    step = 1
    while True:
        try:
            compile(current_code, f"<step_{step}>", "exec")
            print(f"🎉 恭喜！在第 {step} 輪成功通過語法編譯，所有語法錯誤全數清除！")
            break
        except SyntaxError as e:
            print(f"[第 {step} 輪排查] 撞到錯誤！行號: 第 {e.lineno} 行 | 描述: {e.msg}")
            # 模擬人工修復
            lines = current_code.split('\n')
            bad_line = lines[e.lineno - 1]
            print(f"  出錯行內容: {repr(bad_line)}")
            
            if "def sum_even_squares(n)" in bad_line:
                print("  --> 處方：def 函式宣告尾端補上冒號 ':'")
                lines[e.lineno - 1] = "def sum_even_squares(n):"
            elif "for i in range(1, n + 1:" in bad_line:
                print("  --> 處方：range 括號未閉合，補上右括號 ')'")
                lines[e.lineno - 1] = "    for i in range(1, n + 1):"
            
            current_code = '\n'.join(lines)
            step += 1

step_by_step_debugger(buggy_code)


### 13.2.6 語法重點回顧與核心觀念提煉

考場極速修復的三大心理戰略：
1. **不要一次想改五個地方**：永遠只專注修復直譯器當前噴出的那一個錯誤！修完立刻存檔重新跑一次，讓直譯器幫你驗收。
2. **循序推進法**：每次修復都是在推進直譯器的剖析進度。從行號較小處一路修到行號較大處，直到整篇綠燈通行。
3. **保持冷靜**：SyntaxError 是程式設計中最容易解決的錯誤，因為直譯器已經把行號、字元與原因清清楚楚告訴你了，只需冷靜按圖索驥即可！


In [ ]:
# 13.2.6 學生實作練習：考場全方位語法醫生實戰
# 任務說明：實作 fix_apcs_starter_code(broken_code) 函式
# 傳入一段具有語法缺陷的程式碼字串 broken_code
# 修復以下常見語法硬傷：
# 1. 將包含全形冒號 '：' 的行換成半形 ':'
# 2. 將包含全形空白 '\u3000' 換成半形空白 ' '
# 3. 確保 'if x = 10:' 等誤用單等號的比較式換成 '=='
# 回傳修復後的純淨合法程式碼！

def fix_apcs_starter_code(broken_code: str) -> str:
    fixed = broken_code.replace('：', ':').replace('\u3000', ' ')
    lines = fixed.split('\n')
    for i, line in enumerate(lines):
        # 簡單修復 if 內部的單等號
        if "if " in line and " = " in line and " == " not in line:
            lines[i] = line.replace(" = ", " == ")
    return '\n'.join(lines)

# 測試用例
buggy_snippet = "if　x = 10：\n　　print('OK')"
print("修復結果:\n" + fix_apcs_starter_code(buggy_snippet))


In [ ]:
# 13.2.6 單元測試驗證
raw = "if　score = 100：\n　　print('PASS')"
res = fix_apcs_starter_code(raw)
assert '：' not in res
assert '\u3000' not in res
assert '==' in res
# 驗證修復後的程式碼能順利編譯
compile(res, "<student_test>", "exec")
print("🎉 13.2.6 所有測試通過！恭喜你已修成 10 秒鎖定修復 SyntaxError 的除錯大師！")


## 13.2 總結與考場語法除錯速查指南

在本單元中，我們徹底揭開了直譯器編譯期（Compilation Phase）的運行本質，並將考場中最常見的語法錯誤一網打盡。以下為考場必備的語法除錯速查表：

| 語法錯誤類型 | 典型 Traceback 訊息 | 典型錯誤代碼現場 | 10 秒排查與修復秘笈 |
| :--- | :--- | :--- | :--- |
| **漏打冒號** | `SyntaxError: expected ':'` | `if x > 0` / `else` / `def f()` | 在 `if`, `elif`, `else`, `for`, `while`, `def` 結尾強制補上 `:` |
| **上一行漏括號** | `SyntaxError: invalid syntax`（箭頭指在下一行） | 上行 `a = (1 + 2`，下行 `b = 3` | **抬頭看前一行！** 補齊遺失的右括號 `)` 或中括號 `]` |
| **全形空白/標點** | `SyntaxError: invalid character '　'` | 複製講義文字包含全形字元 | 切換英文半形輸入法，將全形空白與中文冒號一鍵取代 |
| **等號誤用** | `SyntaxError: invalid syntax. Maybe you meant '=='?` | `if x = 5:` | 條件判斷一律使用雙等號 `==`，單等號 `=` 僅用於變數賦值 |
| **意外縮排** | `IndentationError: unexpected indent` | 行首不小心按了空白鍵 | 頂格對齊最外層程式碼，縮排嚴格統一為 4 個半形空格 |
| **縮排區塊留空** | `IndentationError: expected an indented block` | 冒號下方只有註解或空行 | 若暫無執行邏輯，務必在縮排區塊內補上 `pass` |

### 🚀 下一步學習指引
當語法結構完全合規後，程式碼便能順利啟動並進入執行階段。然而，在程式跑動的中途，往往會因為數值不合常理（如陣列越界、字串轉整數失敗、查無此鍵、除以零）而突然暴斃！
在下一單元 **13-3《執行時期錯誤（Runtime Error, RE）常見排行榜與崩潰防禦》** 中，我們將直搗 APCS 考場五大執行崩潰殺手，建立密不透風的條件式防守心法！
